<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/05_audio/01_audio_classification_quest_spoken_digits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔊 Quest · Audio classification: teach your computer to hear digits

**Bell Labs, 1952.** Three engineers unveil *Audrey* (Automatic Digit Recognizer): a machine filling a whole rack
of electronics that recognises the spoken digits 0 to 9… if they are spoken by its creator, slowly, with pauses.
Ten years later IBM's *Shoebox* understands 16 English words. Today your phone transcribes any sentence in dozens of
languages. In between: 70 years of audio machine learning.

In this quest you will rebuild Audrey with modern tools, in two ways: the **classical** way (hand-crafted audio
features + a classifier, the state of the art until ~2012) and the **deep learning** way (a small convolutional
neural network that "looks" at images of sound).

**Dataset:** the *Free Spoken Digit Dataset* (FSDD): 3 000 recordings of digits 0-9, by 6 speakers, 50 times each,
recorded at 8 kHz.
**Mission:** recognise the digit spoken by **people the model has never heard**.

| You will practise | Concept |
|---|---|
| Reading audio files | waveform, sampling rate |
| Seeing sound | spectrogram, mel scale, MFCC |
| Honest evaluation | **speaker-independent** split (the group trap!) |
| Classical ML on audio | summary statistics of MFCCs + SVM |
| Deep learning on audio | a CNN on log-mel spectrograms |

🧩 Cells marked `# TODO` are yours.

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "librosa"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display
from IPython.display import Audio, display

quest("Teach your computer to hear digits", chapter="Track 5 · Audio · Classification", xp=300)
DEVICE = get_device()

## 1. Listen to the data 🎧

In [ ]:
root = download_and_extract("https://codeload.github.com/Jakobovski/free-spoken-digit-dataset/zip/refs/heads/master",
                            "fsdd", marker="free-spoken-digit-dataset-master")
files = sorted((root / "free-spoken-digit-dataset-master" / "recordings").glob("*.wav"))
meta = pd.DataFrame({"path": files})
meta[["digit", "speaker", "take"]] = meta["path"].map(lambda p: p.stem.split("_")).tolist()
meta["digit"] = meta["digit"].astype(int)
print(len(meta), "recordings")
meta.groupby("speaker")["digit"].count()

In [ ]:
example = meta.iloc[0]
y, sr = sf.read(example["path"])
print(f"Speaker {example['speaker']} says {example['digit']}")
display(Audio(y, rate=sr))

A recording is just a list of numbers: the air pressure measured `sr` times per second (the **sampling rate**).

### 🎯 Exercise 1: how long is this recording, in seconds?

In [ ]:
duration = ...   # TODO: number of samples divided by the sampling rate
print(f"{len(y)} samples at {sr} Hz = {duration:.2f} s")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
duration = len(y) / sr
print(f"{len(y)} samples at {sr} Hz = {duration:.2f} s")
```

</details>

In [ ]:
check(abs(duration - len(y) / sr) < 1e-9 and 0.1 < duration < 2,
      "Right! At 8 000 samples per second, a digit lasts well under a second.",
      "duration = len(y) / sr")

## 2. Seeing sound: the spectrogram 🌈

The waveform is hard to read. In 1946, Bell Labs (again!) built the *sound spectrograph*: it shows **which
frequencies** are present **at each moment**. It's computed with short Fourier transforms over overlapping windows
(*STFT*). Vowels appear as horizontal bands (*formants*), consonants as noisy bursts.

Humans hear pitch on a roughly logarithmic scale, so we squeeze the frequency axis with the **mel scale** (1937) and
take the logarithm of the energy (decibels), like our ears do.

In [ ]:
N_FFT, HOP, N_MELS = 256, 80, 40       # 32 ms windows, every 10 ms, 40 mel bands

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
librosa.display.waveshow(y, sr=sr, ax=axes[0]); axes[0].set_title(f"Waveform: '{example['digit']}'")
S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)), ref=np.max)
librosa.display.specshow(S, sr=sr, hop_length=HOP, x_axis="time", y_axis="hz", ax=axes[1]); axes[1].set_title("Spectrogram")
M = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS), ref=np.max)
librosa.display.specshow(M, sr=sr, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[2]); axes[2].set_title("Log-mel spectrogram")
plt.tight_layout(); plt.show()

### 🎯 Exercise 2: one log-mel spectrogram per digit
Write `log_mel(y, sr)` returning the log-mel spectrogram (use the 3 constants above), then look at how each digit
"looks".

In [ ]:
def log_mel(y, sr):
    # TODO: librosa.feature.melspectrogram(y=..., sr=..., n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS), then librosa.power_to_db(..., ref=np.max)
    ...

fig, axes = plt.subplots(1, 10, figsize=(18, 2.4))
for d, ax in enumerate(axes):
    yy, _ = sf.read(meta.query("digit == @d and speaker == 'jackson'").iloc[0]["path"])
    ax.imshow(log_mel(yy, sr), origin="lower", aspect="auto", cmap="magma"); ax.set_title(str(d)); ax.axis("off")
plt.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def log_mel(y, sr):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    return librosa.power_to_db(mel, ref=np.max)

fig, axes = plt.subplots(1, 10, figsize=(18, 2.4))
for d, ax in enumerate(axes):
    yy, _ = sf.read(meta.query("digit == @d and speaker == 'jackson'").iloc[0]["path"])
    ax.imshow(log_mel(yy, sr), origin="lower", aspect="auto", cmap="magma"); ax.set_title(str(d)); ax.axis("off")
plt.show()
```

</details>

In [ ]:
m = log_mel(y, sr)
check(m is not None and m.shape[0] == N_MELS and m.max() <= 0,
      f"Spectrogram shape {m.shape}: {N_MELS} mel bands × time frames. 'six' (s-i-ks) and 'seven' look different, right?",
      "Return librosa.power_to_db(librosa.feature.melspectrogram(...), ref=np.max)")
checkpoint("digits/spectrogram", xp=50)

## 3. Classical features: MFCCs 🎛️

**MFCCs** (*Mel-Frequency Cepstral Coefficients*, 1980) compress each time frame of the log-mel spectrogram into ~20
numbers describing the *shape* of the spectrum (the shape of your vocal tract), ignoring the pitch. They ruled speech
recognition for 30 years.

A recording has a variable number of frames, but a classic classifier needs a fixed-size vector: we summarise each
MFCC over time by its **mean** and **standard deviation**.

### 🎯 Exercise 3: MFCC summary features

In [ ]:
def mfcc_features(y, sr, n_mfcc=20):
    mfcc = ...      # TODO: librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=N_FFT, hop_length=HOP) → shape (20, frames)
    return ...      # TODO: concatenate the mean and the std over time (axis=1) → shape (40,)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def mfcc_features(y, sr, n_mfcc=20):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=N_FFT, hop_length=HOP)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])
```

</details>

In [ ]:
f = mfcc_features(y, sr)
check(f is not None and np.shape(f) == (40,), "40 numbers summarise a whole recording.", "Expected a vector of shape (40,).")

audio = [sf.read(p)[0] for p in meta["path"]]                       # load all 3 000 clips (a few seconds)
X = np.stack([mfcc_features(a, sr) for a in audio])
y_digit = meta["digit"].to_numpy()
print(X.shape)

## 4. 🛡️ The honest split: unseen speakers

Our mission is to recognise digits spoken by **new people**. Remember the playbook's group trap: if the same speaker
is in train and test, the model can rely on *their* voice and look better than it is.

### 🎯 Exercise 4: compare a random split with a speaker split
We hold out two speakers for the test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

TEST_SPEAKERS = ["theo", "yweweler"]
is_test = ...                         # TODO: boolean array, True for rows whose speaker is in TEST_SPEAKERS (hint: .isin)

svm = make_pipeline(StandardScaler(), SVC(C=10))

# (a) speaker-independent split
svm.fit(X[~is_test], y_digit[~is_test])
acc_unseen = svm.score(X[is_test], y_digit[is_test])

# (b) random split of the same size
Xa, Xb, ya, yb = train_test_split(X, y_digit, test_size=is_test.mean(), random_state=SEED, stratify=y_digit)
acc_random = ...                      # TODO: fit svm on (Xa, ya) and score it on (Xb, yb)
print(f"Random split: {acc_random:.1%}   |   unseen speakers: {acc_unseen:.1%}")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

TEST_SPEAKERS = ["theo", "yweweler"]
is_test = meta["speaker"].isin(TEST_SPEAKERS).to_numpy()

svm = make_pipeline(StandardScaler(), SVC(C=10))

svm.fit(X[~is_test], y_digit[~is_test])
acc_unseen = svm.score(X[is_test], y_digit[is_test])

Xa, Xb, ya, yb = train_test_split(X, y_digit, test_size=is_test.mean(), random_state=SEED, stratify=y_digit)
acc_random = svm.fit(Xa, ya).score(Xb, yb)
print(f"Random split: {acc_random:.1%}   |   unseen speakers: {acc_unseen:.1%}")
```

</details>

In [ ]:
check(acc_random > acc_unseen,
      "The random split is too optimistic: the model partly recognises voices it has already heard. "
      "Only the speaker split answers our real question.",
      "With a correct is_test mask, the random split should look better than the unseen-speaker split.")
lb = Leaderboard("spoken_digits", metric="accuracy_unseen_speakers")
lb.add("MFCC mean/std + SVM", acc_unseen)
checkpoint("digits/honest_split", xp=60)

## 5. Deep learning: a CNN that looks at spectrograms 🧠

Since ~2012, the best audio models learn their own features. A trick that works remarkably well: treat the log-mel
spectrogram as an **image** and use a **convolutional neural network** (CNN), the same family of models as in the
computer-vision track. Small filters slide over time and frequency and learn to detect bursts, formant transitions…

Every clip is **peak-normalised** (same loudness) and centred in a 1-second window, so that all images have the
same size (40 mel bands × 101 frames). Then we standardise the values with the mean and std of the training set.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def fixed_log_mel(a, length=8000):
    a = a.astype(np.float32) / (np.abs(a).max() + 1e-8)          # same loudness for everyone
    pad = max(0, length - len(a))
    a = np.pad(a, (pad // 2, pad - pad // 2))[:length]            # centre the digit in 1 second
    mel = librosa.feature.melspectrogram(y=a, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    return librosa.power_to_db(mel, ref=1.0, top_db=80)

M = np.stack([fixed_log_mel(a) for a in audio])
M = (M - M[~is_test].mean()) / M[~is_test].std()                  # statistics from the training speakers only!
M = torch.tensor(M, dtype=torch.float32).unsqueeze(1)             # (N, 1, 40, 101): a batch of 1-channel images
T = torch.tensor(y_digit.copy())
test_mask = torch.tensor(is_test.copy())
train_dl = DataLoader(TensorDataset(M[~test_mask], T[~test_mask]), batch_size=64, shuffle=True, num_workers=NUM_WORKERS)
test_dl = DataLoader(TensorDataset(M[test_mask], T[test_mask]), batch_size=256, num_workers=NUM_WORKERS)
print(M.shape)

The architecture: three convolution blocks (each halves the image), then we **average over the frequency axis
but keep 6 time steps**. Why not average everything? Because a digit is a *sequence* of sounds: "seven" is
*s-e-v-e-n*, and a model that forgets the order of the sounds is much weaker (try it in the bonus!).

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))

model = nn.Sequential(
    conv_block(1, 32), conv_block(32, 64), conv_block(64, 128),     # 40×101 → 5×12 feature maps
    nn.AdaptiveAvgPool2d((1, 6)),                                    # average over frequency, keep 6 time steps
    nn.Flatten(), nn.Dropout(0.3), nn.Linear(128 * 6, 10),
).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

### 🎯 Exercise 5: the training step
The four lines at the heart of **every** PyTorch training loop:
1. forward pass: `logits = model(xb)`
2. loss: `loss = loss_fn(logits, yb)`
3. reset old gradients, backward pass: `optimizer.zero_grad()` then `loss.backward()`
4. update the weights: `optimizer.step()`

In [ ]:
EPOCHS = 20
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)   # slowly lower the learning rate

@torch.no_grad()
def evaluate(dl):
    model.eval()
    correct = sum((model(xb.to(DEVICE)).argmax(1).cpu() == yb).sum().item() for xb, yb in dl)
    model.train()
    return correct / len(dl.dataset)

for epoch in range(EPOCHS):
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        ...   # TODO: forward pass, loss, zero_grad, backward, step (4-5 lines)
    scheduler.step()
    if epoch % 4 == 3:
        print(f"epoch {epoch + 1:2d}  train acc {evaluate(train_dl):.1%}  unseen-speaker acc {evaluate(test_dl):.1%}")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
EPOCHS = 20
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)   # slowly lower the learning rate

@torch.no_grad()
def evaluate(dl):
    model.eval()
    correct = sum((model(xb.to(DEVICE)).argmax(1).cpu() == yb).sum().item() for xb, yb in dl)
    model.train()
    return correct / len(dl.dataset)

for epoch in range(EPOCHS):
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()
    if epoch % 4 == 3:
        print(f"epoch {epoch + 1:2d}  train acc {evaluate(train_dl):.1%}  unseen-speaker acc {evaluate(test_dl):.1%}")
```

</details>

In [ ]:
acc_cnn = evaluate(test_dl)
lb.add("CNN on log-mel spectrograms", acc_cnn)
check(acc_cnn > 0.65, "Your CNN understands people it has never heard, and beats the MFCC + SVM model!",
      "Accuracy should exceed 65%. Did you call optimizer.zero_grad(), loss.backward() and optimizer.step()?")
lb.show()

## 6. 🔬 Which digits get confused?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
model.eval()
with torch.no_grad():
    preds = torch.cat([model(xb.to(DEVICE)).argmax(1).cpu() for xb, _ in test_dl]).numpy()
ConfusionMatrixDisplay.from_predictions(y_digit[is_test], preds, cmap="Blues")
plt.title("CNN on unseen speakers"); plt.grid(False); plt.show()

Look for the classic pairs: digits that share sounds ("five"/"nine" share the long *i* sound, "six"/"seven" both
start with *s*). With only 4 training speakers, voices the model has never heard (accent, pitch, microphone) remain
the main challenge: real systems train on thousands of speakers.

## 🏆 Quest complete!

You rebuilt Audrey, and made it work for strangers, which Audrey couldn't.

### 🚀 Your turn (bonus XP)
1. Replace `nn.AdaptiveAvgPool2d((1, 6))` by `nn.AdaptiveAvgPool2d(1)` (and the last layer by `nn.Linear(128, 10)`):
   the model now forgets the order of the sounds. How much accuracy do you lose?
2. **Data augmentation**: add random noise, shift the audio in time, or mask random frequency bands of the
   spectrogram (*SpecAugment*, Google 2019) during training. Does the unseen-speaker accuracy improve?
3. Do a full **leave-one-speaker-out** cross-validation with `GroupKFold(n_splits=6)` for the SVM. Which speaker is
   hardest?
4. Record yourself saying digits with your phone, save the files as `.wav`, and test the model (resample to 8 kHz
   with `librosa.load(path, sr=8000)`).
5. Next: `02_speech_recognition_whisper.ipynb`, from 10 digits to full sentences in many languages.

In [ ]:
checkpoint("digits/complete", xp=190)
badge("Audrey Reborn", "🔊")
progress()